In [1]:
import ray
import pandas as pd
import numpy as np
import modules.networker as netw
import modules.network_extractor as net_extractor
import shapely
from osmnx import settings
import osmnx as ox
import networkx as nx

In [2]:
ray.shutdown()
ray.init()

2025-03-25 15:06:43,627	INFO worker.py:1832 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Python version:,3.11.3
Ray version:,2.42.1
Dashboard:,http://127.0.0.1:8265


In [ ]:
# Add here your path 
data_base_path = "/home/user/Desktop/JP/street-network-indices/data"
# The extractor instance
extractor = net_extractor.NetworkExtractor(base_path=data_base_path)
 
settings.default_crs = "epsg:4326"
#box = shapely.box(-75.709319,45.407791,-75.678506,45.419118)
#box = shapely.box(9.165816,45.468242,9.172597,45.472110)
#geom = box
#geom = shapely.box(-75.687502,45.427243,-75.685024,45.428346)
#geom = shapely.box(-75.767727,45.337667,-75.709705,45.360830)

#mostar
# geom = shapely.box(17.775965, 43.324428, 17.833214, 43.372176)

# Milan
geom = shapely.box(9.040613,45.386725,9.277997,45.535947)


In [213]:
# General info to get queries
geometry = geom
place = "milan"
place_DTM = "Milan"
dist_threshold=20
slope_threshold=15
assess = False

In [214]:
# Obtain relations
import requests

aoi = ox._overpass._make_overpass_polygon_coord_strs(geom)[0]
overpass_query = f'[out:json][timeout:60];(rel(poly:{aoi!r})[\"route\"~\"train|subway|monorail|tram|bus|trolleybus|ferry\"];rel(r)(poly:{aoi!r}););out;'
overpass_url = "https://overpass-api.de/api/interpreter"

print(overpass_query)

body = {
    "data": overpass_query
}
r = requests.post(overpass_url, body)
relations = r.json()["elements"]

[out:json][timeout:60];(rel(poly:'45.386725 9.277997 45.535947 9.277997 45.535947 9.040613 45.386725 9.040613 45.386725 9.277997')["route"~"train|subway|monorail|tram|bus|trolleybus|ferry"];rel(r)(poly:'45.386725 9.277997 45.535947 9.277997 45.535947 9.040613 45.386725 9.040613 45.386725 9.277997'););out;


In [215]:
# Obtain nodes
tags = {'public_transport': ['stop_position','platform']}
gdfox = ox.features_from_polygon(geom, tags)
gdfox = gdfox.reset_index()
gdfox = gdfox.loc[gdfox["element"] == "node"]
gdfox = gdfox.set_index("id")
nodes_dict = gdfox.to_dict(orient="index")

In [216]:
public_graph = nx.MultiDiGraph()

for n,node_data in nodes_dict.items():
    data = {
        "name": node_data['name'] if "name" in node_data else None,
        "x": node_data['geometry'].x,
        "y": node_data['geometry'].y,

        "train": node_data['train'] if "train" in node_data else None,
        "subway": node_data['subway'] if "subway" in node_data else None,
        "monorail": node_data['monorail'] if "monorail" in node_data else None,
        "tram": node_data['tram'] if "tram" in node_data else None,
        "bus": node_data['bus'] if "bus" in node_data else None,
        "trolleybus": node_data['trolleybus'] if "trolleybus" in node_data else None,
        "ferry": node_data['ferry'] if "ferry" in node_data else None,
    }
    public_graph.add_node(n, **data)

In [217]:

for rel in relations:
    rel_metadata = rel["tags"]
    node_list = list(filter(lambda x: x["type"] != "way" and (x["role"] == "stop" or x["role"] == "platform"), rel["members"]))
    
    prev_stop = None
    for node in node_list:
        ref = node["ref"]
        if ref in nodes_dict:
            if prev_stop is None:
                # first node
                prev_stop = ref
            else:
                edge_data = {
                    "geometry": shapely.LineString([nodes_dict[prev_stop]["geometry"], nodes_dict[ref]["geometry"]]),
                    "description": rel_metadata['description'] if "description" in rel_metadata else None,
                    "from": rel_metadata['from'] if "from" in rel_metadata else None,
                    "to": rel_metadata['to'] if "to" in rel_metadata else None,
                    "operator": rel_metadata['operator'] if "operator" in rel_metadata else None,
                    "network": rel_metadata['network'] if "network" in rel_metadata else None,
                    "type": rel_metadata['type'] if "type" in rel_metadata else None,
                    "route": rel_metadata['route'] if "route" in rel_metadata else None
                }
                u = prev_stop
                v = ref
                public_graph.add_edge(u,v, **edge_data)
                prev_stop = ref

        else:
            prev_stop = None

public_graph.graph["crs"] = "epsg:4326"

In [218]:

extractor.save_as_shp(public_graph, f'{place}/shp/public_{place}')

/home/geolab/Desktop/Research/notebooks/modules/network_extractor.py:189: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  edges.to_file(f"{self.DATA_BASE_PATH}/{path}_edges.shp", encoding='utf-8')
/home/geolab/Desktop/Research/.venv/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'description' to 'descriptio'
  ogr_write(


In [ ]:
"""
Extract a OSMnx network from 
Types: drive | bike | walk 

TODO: public transport network.
""" 
geometry = geom
place = "milan"
place_DTM = "Milan"
dist_threshold=20
slope_threshold=15
assess = False

g_promises = []
g_promises.append(extractor.download_network.remote(
    extractor,
    "walk", 
    geometry, 
    place_DTM, 
    assessment=assess, 
    dist_threshold=dist_threshold, 
    slope_threshold=slope_threshold,
    add_elevation=True,
))

g_promises.append(extractor.download_network.remote(
    extractor,
    "bike", 
    geometry, 
    place_DTM, 
    assessment=assess, 
    add_elevation=True,
))

g_promises.append(extractor.download_network.remote(
    extractor,
    "drive", 
    geometry, 
    place_DTM, 
    assessment=assess, 
    add_elevation=True,
))

g_promises.append(extractor.download_network.remote(
    extractor,
    "drive", 
    geometry, 
    place_DTM, 
    assessment=assess, 
    add_elevation=True,
))

[g_walk, g_bike, g_drive] = ray.get(g_promises)

extractor.save_as_shp(g_walk, f'{place}/shp/walk_{place}')
extractor.save_as_shp(g_bike, f'{place}/shp/bike_{place}')
extractor.save_as_shp(g_drive, f'{place}/shp/drive_{place}')

/home/geolab/Desktop/Research/notebooks/modules/network_extractor.py:180: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  nodes.to_file(f"{self.DATA_BASE_PATH}/{path}_nodes.shp", encoding='utf-8')
/home/geolab/Desktop/Research/.venv/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'street_count' to 'street_cou'
  ogr_write(
/home/geolab/Desktop/Research/notebooks/modules/network_extractor.py:182: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  edges.to_file(f"{self.DATA_BASE_PATH}/{path}_edges.shp", encoding='utf-8')
/home/geolab/Desktop/Research/.venv/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'travel_time' to 'travel_tim'
  ogr_write(
/home/geolab/Desktop/Research/notebooks/modules/network_extractor.py:180: UserWarning: Column names longer than 10 characters will be truncated

In [ ]:
"""
Extract a OSMnx network from 
Types: drive | bike | walk 

TODO: public transport network.
""" 
geometry = geom
place = "milan"
place_DTM = "Milan"
dist_threshold=20
slope_threshold=15
assess = False

g_promises = []
g_promises.append(extractor.download_network.remote(
    extractor,
    "walk", 
    geometry, 
    place_DTM, 
    assessment=assess, 
    dist_threshold=dist_threshold, 
    slope_threshold=slope_threshold,
    add_elevation=True,
))

g_promises.append(extractor.download_network.remote(
    extractor,
    "bike", 
    geometry, 
    place_DTM, 
    assessment=assess, 
    add_elevation=True,
))

g_promises.append(extractor.download_network.remote(
    extractor,
    "drive", 
    geometry, 
    place_DTM, 
    assessment=assess, 
    add_elevation=True,
))

[g_walk, g_bike, g_drive] = ray.get(g_promises)

extractor.save_as_shp(g_walk, f'{place}/shp/walk_{place}')
extractor.save_as_shp(g_bike, f'{place}/shp/bike_{place}')
extractor.save_as_shp(g_drive, f'{place}/shp/drive_{place}')

In [ ]:
source = 7545571877
destination = 7414901775

graph: nx.MultiGraph
graph = g_walk

path = nx.shortest_path(graph, source, destination, weight="travel_time")
time_path = nx.shortest_path_length(graph, source, destination, weight="travel_time")

ox.plot_graph_route(graph, path)